In [1]:
import pandas as pd
import numpy as np
import math
import warnings
import string
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerLine2D
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, cross_val_score, KFold
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
warnings.filterwarnings("ignore")

<table class="table my-4 w-full"><thead><tr><th>Variable Name</th><th>Role</th><th>Type</th><th>Demographic</th><th>Description</th><th>Units</th><th>Missing Values</th></tr></thead> <tbody><tr><td>age</td><td>Feature</td><td>Integer</td><td>Age</td><td></td><td>years</td><td>no</td> </tr><tr><td>sex</td><td>Feature</td><td>Categorical</td><td>Sex</td><td></td><td></td><td>no</td> </tr><tr><td>cp</td><td>Feature</td><td>Categorical</td><td></td><td></td><td></td><td>no</td> </tr><tr><td>trestbps</td><td>Feature</td><td>Integer</td><td></td><td>resting blood pressure (on admission to the hospital)</td><td>mm Hg</td><td>no</td> </tr><tr><td>chol</td><td>Feature</td><td>Integer</td><td></td><td>serum cholestoral</td><td>mg/dl</td><td>no</td> </tr><tr><td>fbs</td><td>Feature</td><td>Categorical</td><td></td><td>fasting blood sugar &gt; 120 mg/dl</td><td></td><td>no</td> </tr><tr><td>restecg</td><td>Feature</td><td>Categorical</td><td></td><td></td><td></td><td>no</td> </tr><tr><td>thalach</td><td>Feature</td><td>Integer</td><td></td><td>maximum heart rate achieved</td><td></td><td>no</td> </tr><tr><td>exang</td><td>Feature</td><td>Categorical</td><td></td><td>exercise induced angina</td><td></td><td>no</td> </tr><tr><td>oldpeak</td><td>Feature</td><td>Integer</td><td></td><td>ST depression induced by exercise relative to rest</td><td></td><td>no</td> </tr><tr><td>slope</td><td>Feature</td><td>Categorical</td><td></td><td></td><td></td><td>no</td> </tr><tr><td>ca</td><td>Feature</td><td>Integer</td><td></td><td>number of major vessels (0-3) colored by flourosopy</td><td></td><td>yes</td> </tr><tr><td>thal</td><td>Feature</td><td>Categorical</td><td></td><td></td><td></td><td>yes</td> </tr><tr><td>num</td><td>Target</td><td>Integer</td><td></td><td>diagnosis of heart disease</td><td></td><td>no</td> </tr></tbody></table>

In [2]:
pcData = pd.read_csv("./processed.cleveland.data", delimiter= ",", header=None)

Having a look at the data

In [3]:
pcData

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1


In [4]:
for col in pcData.columns:
    print(f"\nUnique values in column {col}:")
    print(pcData[col].unique())


Unique values in column 0:
[63. 67. 37. 41. 56. 62. 57. 53. 44. 52. 48. 54. 49. 64. 58. 60. 50. 66.
 43. 40. 69. 59. 42. 55. 61. 65. 71. 51. 46. 45. 39. 68. 47. 34. 35. 29.
 70. 77. 38. 74. 76.]

Unique values in column 1:
[1. 0.]

Unique values in column 2:
[1. 4. 3. 2.]

Unique values in column 3:
[145. 160. 120. 130. 140. 172. 150. 110. 132. 117. 135. 112. 105. 124.
 125. 142. 128. 170. 155. 104. 180. 138. 108. 134. 122. 115. 118. 100.
 200.  94. 165. 102. 152. 101. 126. 174. 148. 178. 158. 192. 129. 144.
 123. 136. 146. 106. 156. 154. 114. 164.]

Unique values in column 4:
[233. 286. 229. 250. 204. 236. 268. 354. 254. 203. 192. 294. 256. 263.
 199. 168. 239. 275. 266. 211. 283. 284. 224. 206. 219. 340. 226. 247.
 167. 230. 335. 234. 177. 276. 353. 243. 225. 302. 212. 330. 175. 417.
 197. 198. 290. 253. 172. 273. 213. 305. 216. 304. 188. 282. 185. 232.
 326. 231. 269. 267. 248. 360. 258. 308. 245. 270. 208. 264. 321. 274.
 325. 235. 257. 164. 141. 252. 255. 201. 222. 260. 182. 303.

In [5]:
print(pcData.columns.tolist())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


In [6]:
columns_to_check = [11, 12]

data = pcData[columns_to_check].values

for i, col in enumerate(columns_to_check):
    count = np.sum(data[:, i] == '?')
    print(f"Missing entries in '{col}': {count}")

Missing entries in '11': 4
Missing entries in '12': 2


In [7]:
rows_with_missing = {}

for i, col in enumerate(columns_to_check):
    rows = np.where(data[:, i] == '?')[0]
    rows_with_missing[col] = rows

In [8]:
print(rows_with_missing[11].tolist())
print(rows_with_missing[12].tolist())

[166, 192, 287, 302]
[87, 266]


In [9]:
pcData[[11, 12]] = pcData[[11, 12]].replace('?', np.nan).astype(float)

pcData['col11_missing'] = pcData[11].isna().astype(int)
pcData['col12_missing'] = pcData[12].isna().astype(int)

iter_imputer = IterativeImputer(random_state=0)
pcData[['col11_iter_imputed', 'col12_iter_imputed']] = iter_imputer.fit_transform(pcData[[11, 12]])

valid_ca_values = [0.0, 1.0, 2.0, 3.0]
valid_thal_values = [3.0, 6.0, 7.0]

def closest_category(x, valid_values):
    return min(valid_values, key=lambda val: abs(val - x))

pcData['ca_final'] = pcData['col11_iter_imputed'].apply(lambda x: closest_category(x, valid_ca_values))
pcData['thal_final'] = pcData['col12_iter_imputed'].apply(lambda x: closest_category(x, valid_thal_values))

print("Imputed values for Column 11 (ca):")
print(pcData.loc[pcData['col11_missing'] == 1, [11, 'col11_iter_imputed', 'ca_final']])

print("\nImputed values for Column 12 (thal):")
print(pcData.loc[pcData['col12_missing'] == 1, [12, 'col12_iter_imputed', 'thal_final']])

pcData[11] = pcData['ca_final']
pcData[12] = pcData['thal_final']

pcData.drop(columns=['col11_iter_imputed', 'col12_iter_imputed', 'ca_final', 'thal_final'], inplace=True)

Imputed values for Column 11 (ca):
     11  col11_iter_imputed  ca_final
166 NaN            0.467182       0.0
192 NaN            0.941760       1.0
287 NaN            0.941760       1.0
302 NaN            0.467182       0.0

Imputed values for Column 12 (thal):
     12  col12_iter_imputed  thal_final
87  NaN            4.388078         3.0
266 NaN            4.388078         3.0


In [10]:
pcData

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,col11_missing,col12_missing
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,0,0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,0,0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1,0,0
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2,0,0
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3,0,0
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1,0,0


In [11]:
new_column_names = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num",
    "ca_missing",
    "thal_missing"
]

In [12]:
pcData.columns = new_column_names

In [13]:
pcData

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,ca_missing,thal_missing
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,0,0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,0,0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1,0,0
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2,0,0
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3,0,0
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1,0,0


In [14]:
pcData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           303 non-null    float64
 1   sex           303 non-null    float64
 2   cp            303 non-null    float64
 3   trestbps      303 non-null    float64
 4   chol          303 non-null    float64
 5   fbs           303 non-null    float64
 6   restecg       303 non-null    float64
 7   thalach       303 non-null    float64
 8   exang         303 non-null    float64
 9   oldpeak       303 non-null    float64
 10  slope         303 non-null    float64
 11  ca            303 non-null    float64
 12  thal          303 non-null    float64
 13  num           303 non-null    int64  
 14  ca_missing    303 non-null    int64  
 15  thal_missing  303 non-null    int64  
dtypes: float64(13), int64(3)
memory usage: 38.0 KB


In [15]:
print(pcData.columns.tolist())

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num', 'ca_missing', 'thal_missing']


In [16]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'ca_missing', 'thal_missing']

for col in cat_cols:
    pcData[col] = pcData[col].astype(float).astype(int).astype('category')

In [17]:
pcData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   age           303 non-null    float64 
 1   sex           303 non-null    category
 2   cp            303 non-null    category
 3   trestbps      303 non-null    float64 
 4   chol          303 non-null    float64 
 5   fbs           303 non-null    category
 6   restecg       303 non-null    category
 7   thalach       303 non-null    float64 
 8   exang         303 non-null    category
 9   oldpeak       303 non-null    float64 
 10  slope         303 non-null    category
 11  ca            303 non-null    float64 
 12  thal          303 non-null    category
 13  num           303 non-null    int64   
 14  ca_missing    303 non-null    category
 15  thal_missing  303 non-null    category
dtypes: category(9), float64(6), int64(1)
memory usage: 20.5 KB


In [18]:
pcData.describe(include='all')

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,ca_missing,thal_missing
count,303.000000,303.0,303.0,303.000000,303.000000,303.0,303.0,303.000000,303.0,303.000000,303.0,303.000000,303.0,303.000000,303.0,303.0
unique,NaN,2.0,4.0,NaN,NaN,2.0,3.0,NaN,2.0,NaN,3.0,NaN,3.0,NaN,2.0,2.0
top,NaN,1.0,4.0,NaN,NaN,0.0,0.0,NaN,0.0,NaN,1.0,NaN,3.0,NaN,0.0,0.0
freq,NaN,206.0,144.0,NaN,NaN,258.0,151.0,NaN,204.0,NaN,142.0,NaN,168.0,NaN,299.0,301.0
mean,54.438944,NaN,NaN,131.689769,246.693069,NaN,NaN,149.607261,NaN,1.039604,NaN,0.669967,NaN,0.937294,NaN,NaN
std,9.038662,NaN,NaN,17.599748,51.776918,NaN,NaN,22.875003,NaN,1.161075,NaN,0.933193,NaN,1.228536,NaN,NaN
min,29.000000,NaN,NaN,94.000000,126.000000,NaN,NaN,71.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN
25%,48.000000,NaN,NaN,120.000000,211.000000,NaN,NaN,133.500000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN
50%,56.000000,NaN,NaN,130.000000,241.000000,NaN,NaN,153.000000,NaN,0.800000,NaN,0.000000,NaN,0.000000,NaN,NaN
75%,61.000000,NaN,NaN,140.000000,275.000000,NaN,NaN,166.000000,NaN,1.600000,NaN,1.000000,NaN,2.000000,NaN,NaN


In [19]:
def encode_categorical_features(df, categorical_cols, drop_first=True, sparse=False):
    encoder = OneHotEncoder(drop='first' if drop_first else None,
                            sparse_output=sparse,
                            handle_unknown='ignore')
    encoded_array = encoder.fit_transform(df[categorical_cols])
    encoded_df = pd.DataFrame(encoded_array,
                              columns=encoder.get_feature_names_out(categorical_cols),
                              index=df.index)
    
    df_final = pd.concat([df.drop(columns=categorical_cols), encoded_df], axis=1)
    
    return df_final

In [20]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'ca_missing', 'thal_missing']

df_encoded= encode_categorical_features(pcData, categorical_cols)

In [21]:
df_encoded

,age,trestbps,chol,thalach,oldpeak,ca,num,sex_1,cp_2,cp_3,...,fbs_1,restecg_1,restecg_2,exang_1,slope_2,slope_3,thal_6,thal_7,ca_missing_1,thal_missing_1
0,63.0,145.0,233.0,150.0,2.3,0.0,0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1,67.0,160.0,286.0,108.0,1.5,3.0,2,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,67.0,120.0,229.0,129.0,2.6,2.0,1,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,37.0,130.0,250.0,187.0,3.5,0.0,0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,41.0,130.0,204.0,172.0,1.4,0.0,0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,110.0,264.0,132.0,1.2,0.0,1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
299,68.0,144.0,193.0,141.0,3.4,2.0,2,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
300,57.0,130.0,131.0,115.0,1.2,1.0,3,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
301,57.0,130.0,236.0,174.0,0.0,1.0,1,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [22]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             303 non-null    float64
 1   trestbps        303 non-null    float64
 2   chol            303 non-null    float64
 3   thalach         303 non-null    float64
 4   oldpeak         303 non-null    float64
 5   ca              303 non-null    float64
 6   num             303 non-null    int64  
 7   sex_1           303 non-null    float64
 8   cp_2            303 non-null    float64
 9   cp_3            303 non-null    float64
 10  cp_4            303 non-null    float64
 11  fbs_1           303 non-null    float64
 12  restecg_1       303 non-null    float64
 13  restecg_2       303 non-null    float64
 14  exang_1         303 non-null    float64
 15  slope_2         303 non-null    float64
 16  slope_3         303 non-null    float64
 17  thal_6          303 non-null    flo